# Annular Boussinesq — AWFNO capacity study (width 32 → 40)

**Based on:** `annulus2d_jcp_one_run_all.ipynb`, using the **original sample-size AWFNO benchmark**. This is a capacity-only change, not a new architecture or training protocol.

**Change:** only the AWFNO latent width is increased from **32 to 40**; TB-AWFNO, WFNO and FNO retain their existing architectures and can reuse their original checkpoints. The original settings for the dataset, splits, $K=30$, local radius $r=2$, Fourier modes $(16,24)$, 4 blocks, optimizer, and evaluation remain unchanged.

**One run:** In Google Colab, select a GPU, then **Runtime → Run all**. Drive will be mounted and the experiment will execute for $N_{\mathrm{train}}=25,50,100,200,400$ by default. You may change the sizes in the settings cell.

**Separate outputs:** Original width-32 plots, results and checkpoints are preserved. New results are under `MyDrive/anus/output/cache_annulus2d_awfno_sample_sweep_sizes25-50-100-200-400_K30_M18_W32_mr16_mt24_R2/awfno_capacity_W40/`. The code reuses original width-32 baseline checkpoints when available; new W40 AWFNO weights are saved with `_W40_` in the checkpoint name.

**Reported metrics:** physical-annulus absolute space–time and final-time $L^2$, saved rollout arrays, and true/predicted/error plots. The notebook does **not** claim to compute $H^1$: that requires a separate spatial-derivative diagnostic.


In [ ]:
#@title 1. Mount Drive and verify the source data
from google.colab import drive
from pathlib import Path
import os, sys, subprocess, json, time
import torch

drive.mount('/content/gdrive')
DATASET = Path('/content/gdrive/MyDrive/anus/hyteg_annulus_c_grid_64x128_samples500.pt')
OUTPUT_ROOT = Path('/content/gdrive/MyDrive/anus/output')
CONTROLLED_OUTPUT = OUTPUT_ROOT / 'jcp_controlled_annulus'
if not DATASET.is_file():
    root = DATASET.parent
    if root.is_dir():
        print('Candidates:', [p.name for p in root.iterdir() if 'annulus' in p.name.lower()])
    raise FileNotFoundError('Expected annulus .pt dataset not found: '+str(DATASET))
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
print('Data:', DATASET, 'size GiB:',round(DATASET.stat().st_size/1024**3,3))
print('Device:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if not torch.cuda.is_available():
    raise RuntimeError('Set Colab runtime to GPU before starting the full experimental suite.')


In [ ]:
#@title 2. AWFNO capacity sweep settings
SAMPLE_SIZES = '25,50,100,200,400' #@param {type:'string'}
AWFNO_WIDTH = 40 #@param {type:'integer'}
EPOCHS = 150 #@param {type:'integer'}
BATCH_SIZE = 32 #@param {type:'integer'}
FORCE_RETRAIN = False #@param {type:'boolean'}

assert AWFNO_WIDTH >= 1, 'AWFNO_WIDTH must be positive'
assert SAMPLE_SIZES, 'Select one or more sample sizes'
assert all(int(x.strip()) in (25,50,100,200,400) for x in SAMPLE_SIZES.split(',')), 'Allowed N_train: 25,50,100,200,400'
print('AWFNO width:', AWFNO_WIDTH, '| unchanged TB-AWFNO / WFNO width: 32')
print('Training sizes:', SAMPLE_SIZES, '| epochs:', EPOCHS, '| batch:', BATCH_SIZE)
print('Source data:', DATASET)
print('Original checkpoint root:', OUTPUT_ROOT)


In [ ]:
#@title 3. Restore AWFNO-width-40 Python driver (self-contained)
import base64, gzip, hashlib
DRIVER = Path('/content/Boussinesq2D_JCP_AWFNO_capacity_driver.py')
packed = """..."""
payload = gzip.decompress(base64.b64decode(packed))
assert hashlib.sha256(payload).hexdigest() == 'fe23ace75620055b11046a44b2ef23d62c9282b84be9f9dc033170c9238267d1', 'Embedded driver checksum mismatch'
DRIVER.write_bytes(payload)
print('Driver:', DRIVER, '| bytes:', len(payload))


In [ ]:
#@title 4. Install FNO baseline dependency (as in original notebook)
# The reference FNO uses neuralop.models.FNO, retained unchanged.
try:
    from neuralop.models import FNO
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'neuraloperator'])
    from neuralop.models import FNO
print('FNO dependency available')


In [ ]:
#@title 5. RUN — sample-size study (AWFNO width increased, other models unchanged)
# Finished checkpoints are kept in Google Drive; a rerun will reuse them.
cfg = os.environ.copy()
cfg.update({
    'DATA_PATH':str(DATASET),
    'OUTPUT_ROOT':str(OUTPUT_ROOT),
    'EXPERIMENT_SUITE':'original',
    'RUN_SAMPLE_SIZES':SAMPLE_SIZES,
    'AWFNO_WIDTH':str(AWFNO_WIDTH),
    'WIDTH':'32',
    'EPOCHS':str(EPOCHS),
    'BATCH_SIZE':str(BATCH_SIZE),
    'FORCE_RETRAIN':'1' if FORCE_RETRAIN else '0',
    'PYTHONUNBUFFERED':'1',
})
print('Starting capacity comparison for N_train =', SAMPLE_SIZES, flush=True)
subprocess.run([sys.executable,'-u',str(DRIVER)], env=cfg, check=True)
print('Capacity study complete. Results and checkpoints are on Google Drive.')


In [ ]:
#@title 6. View AWFNO capacity results
import pandas as pd
from IPython.display import display
root = OUTPUT_ROOT / 'cache_annulus2d_awfno_sample_sweep_sizes25-50-100-200-400_K30_M18_W32_mr16_mt24_R2'
results_dir = root / f'awfno_capacity_W{AWFNO_WIDTH}' / 'results'
progress = results_dir / 'sample_size_sweep_progress.json'
if progress.exists():
    df = pd.DataFrame(json.loads(progress.read_text())).sort_values('N_train')
    columns = ['N_train','AWFNO_width','AWFNO_parameters',
               'AWFNO_full_l2_mean','AWFNO_final_l2_mean',
               'TB_AWFNO_full_l2_mean','WFNO_full_l2_mean','FNO_full_l2_mean',
               'AWFNO_train_time_sec','rollout_path']
    display(df[[c for c in columns if c in df.columns]])
    print('New results:', results_dir)
    print('New plots:', results_dir.parent / 'plots')
    print('Original results and plots remain in:', root)
else:
    print('No completed capacity runs yet. Check the previous cell or Drive mount:', progress)
